In [ ]:
import os
import pandas as pd
import plotly.graph_objects as go
from typing import List, Optional, Dict
import numpy as np

# --- Configuration ---

# Choose what to plot: 'avg' for mean with std dev, or 'max' for maximum value
plot_metric = 'max'  # Can be 'avg' or 'max'

# Define the workload name to construct the CSV file path
workload_name = "all_to_all" 

# Define the base folder containing all the run directories
base_run_folder = '/app/astra-sim/upc/output/comparison_run/FoldedClos/toy_all_to_all_one_collective'

# --- Helper functions (can be reused from other cells if already defined) ---

def find_config_file(folder_path: str) -> Optional[str]:
    """
    Finds a configuration file (ending with .txt) within the 'configs' subfolder.
    """
    config_dir = os.path.join(folder_path, 'configs')
    if not os.path.isdir(config_dir):
        return None
    for item in os.listdir(config_dir):
        if item.endswith('.txt'):
            return os.path.join(config_dir, item)
    return None

def parse_config(file_path: str) -> Dict[str, str]:
    """
    Parses a 'key = value' or 'key value' configuration file into a dictionary.
    """
    params = {}
    try:
        with open(file_path, 'r') as f:
            for line in f:
                line = line.strip()
                if not line or line.startswith('#'):
                    continue
                parts = line.split('=', 1) if '=' in line else line.split(None, 1)
                if len(parts) == 2:
                    key, value = parts
                    params[key.strip().lower()] = value.strip()
    except FileNotFoundError:
        print(f"Config file not found: {file_path}")
    except Exception as e:
        print(f"Error parsing config file {file_path}: {e}")
    return params

# --- Main Analysis Logic ---

run_folders = [os.path.join(base_run_folder, d) for d in os.listdir(base_run_folder) if os.path.isdir(os.path.join(base_run_folder, d))]

results = []
run_name_counts = {}

for folder in sorted(run_folders):
    print(f"Processing: {os.path.basename(folder)}")

    # 1. Create a descriptive name from the config file
    config_file = find_config_file(folder)
    if not config_file:
        print(f"  - Skipping: Config file not found.")
        continue
    config_params = parse_config(config_file)
    
    # Extract specific parameters for new columns
    ecmp_seed = config_params.get('ecmp_seed', 'N/A')
    use_precomputed_routes = config_params.get('use_precomputed_routes', 'N/A')

    base_run_name = (
        f"cc:{config_params.get('cc_mode', 'N/A')}, "
        f"win:{config_params.get('has_win', 'N/A')}, "
        f"adapt:{config_params.get('var_win', 'N/A')}, "
        f"buf:{config_params.get('buffer_size', 'N/A')}, "
        f"size:{config_params.get('packet_payload_size', 'N/A')}, "
        f"seed:{ecmp_seed}, "
        f"pr:{use_precomputed_routes}"
    )
    
    count = run_name_counts.get(base_run_name, 0)
    run_name_counts[base_run_name] = count + 1
    run_name = f"{base_run_name}_{count}" if count > 0 else base_run_name

    # Initialize result dictionary
    result_entry = {
        'run_name': run_name,
        'avg_elapsed_time': np.nan,
        'std_dev': np.nan,
        'max_elapsed_time': np.nan,
        'ecmp_seed': ecmp_seed,
        'use_precomputed_routes': use_precomputed_routes,
        'folder_name': os.path.basename(folder)
    }

    # 2. Read and filter the timing results CSV file
    timing_file = os.path.join(folder, 'ns3', f'{workload_name}_trace_matched_timing.csv')
    if not os.path.exists(timing_file):
        print(f"  - Skipping plot: Timing file not found at {timing_file}")
        results.append(result_entry)
        continue
        
    try:
        df = pd.read_csv(timing_file)
        # Filter out dummy nodes as they are not part of the actual workload measurement
        df = df[df['node_name'] != 'dummy_node'].copy()
        
        if 'elapsed_time' not in df.columns:
            print(f"  - Skipping plot: 'elapsed_time' column not found in {timing_file}")
            results.append(result_entry)
            continue
    except Exception as e:
        print(f"  - Skipping plot: Failed to read or process {timing_file}: {e}")
        results.append(result_entry)
        continue

    # 3. Calculate statistics for elapsed_time
    elapsed_times = df['elapsed_time'].dropna()
    if not elapsed_times.empty:
        result_entry['avg_elapsed_time'] = elapsed_times.mean()
        result_entry['std_dev'] = elapsed_times.std()
        result_entry['max_elapsed_time'] = elapsed_times.max()
    else:
        print(f"  - Warning: No 'elapsed_time' data found after filtering in {timing_file}")
    
    results.append(result_entry)

# --- Plotting the results ---

if results:
    results_df = pd.DataFrame(results)
    
    # Create a filtered dataframe for plotting that only includes runs with valid timing data
    plot_df = results_df.dropna(subset=['avg_elapsed_time', 'max_elapsed_time']).copy()

    # Convert relevant columns to numeric for filtering
    plot_df['ecmp_seed'] = pd.to_numeric(plot_df['ecmp_seed'], errors='coerce')
    plot_df['use_precomputed_routes'] = pd.to_numeric(plot_df['use_precomputed_routes'], errors='coerce')

    if not plot_df.empty:
        # Define the plot groups and their benchmarks
        plot_groups = [
            {
                "name": "PR=0, Seed=25",
                "filter": (plot_df['use_precomputed_routes'] == 0) & (plot_df['ecmp_seed'] == 25),
                "benchmark": 10119040020
            },
            {
                "name": "PR=0, Seed=1337",
                "filter": (plot_df['use_precomputed_routes'] == 0) & (plot_df['ecmp_seed'] == 1337),
                "benchmark": 10625002645
            },
            {
                "name": "PR=0, Seed=42",
                "filter": (plot_df['use_precomputed_routes'] == 0) & (plot_df['ecmp_seed'] == 42),
                "benchmark": 11130955150
            },
            {
                "name": "PR=1",
                "filter": (plot_df['use_precomputed_routes'] == 1),
                "benchmark": 7589281917
            }
        ]

        # Determine which column to use for plotting
        if plot_metric == 'max':
            y_col = 'max_elapsed_time'
            y_axis_title = "Maximum Elapsed Time (ns)"
            base_plot_title = f'Maximum Elapsed Time Comparison for Workload: "{workload_name}"'
            error_y_config = None
        else: # Default to 'avg'
            y_col = 'avg_elapsed_time'
            y_axis_title = "Average Elapsed Time (ns)"
            base_plot_title = f'Average Elapsed Time Comparison for Workload: "{workload_name}"'
            error_y_config_key = 'std_dev'

        for group in plot_groups:
            group_df = plot_df[group['filter']].copy()

            if group_df.empty:
                print(f"\nNo data to plot for group: {group['name']}")
                continue

            group_df = group_df.sort_values(by=y_col).reset_index(drop=True)

            # Create the bar plot
            fig = go.Figure()
            fig.add_trace(go.Bar(
                x=group_df['run_name'],
                y=group_df[y_col],
                error_y=dict(type='data', array=group_df[error_y_config_key], visible=True) if plot_metric == 'avg' else None,
                marker_color='rgb(55, 83, 109)',
                text=group_df[y_col].apply(lambda x: f'{x/1e9:.4f} s'),
                textposition='outside',
                name='Runs'
            ))

            # Add benchmark line
            fig.add_hline(
                y=group['benchmark'], 
                line_dash="dot",
                annotation_text=f"Benchmark: {group['benchmark']/1e9:.4f} s", 
                annotation_position="bottom right",
                line_color="red"
            )

            fig.update_layout(
                title=f"{base_plot_title} ({group['name']})",
                xaxis_title="Run Configuration",
                yaxis_title=y_axis_title,
                xaxis={'tickangle': -45},
                template='plotly_white',
                height=700,
                width=1200,
                margin=dict(b=250), # Increase bottom margin for long labels
                showlegend=False
            )
            
            print(f"\n--- Plot for group: {group['name']} ---")
            fig.show()
    else:
        print("\nNo runs with valid timing data found to plot.")

    # Display the statistics table for ALL runs
    print("\n--- Summary Statistics (All Runs) ---")
    display(results_df)

else:
    print("\nNo results were processed. Cannot generate plot.")


Processing: run_20251116_220401
Processing: run_20251116_220435
  - Skipping plot: Timing file not found at /app/astra-sim/upc/output/comparison_run/FoldedClos/toy_all_to_all_one_collective/run_20251116_220435/ns3/all_to_all_trace_matched_timing.csv
Processing: run_20251116_220935
Processing: run_20251116_221008
Processing: run_20251116_221043
Processing: run_20251116_221116
Processing: run_20251116_221151
Processing: run_20251116_221225
Processing: run_20251116_221259
Processing: run_20251116_221330
Processing: run_20251116_221401
Processing: run_20251116_221433
Processing: run_20251116_221504
Processing: run_20251116_221537
Processing: run_20251116_221612
Processing: run_20251116_221642
Processing: run_20251116_221711
Processing: run_20251116_221742
Processing: run_20251116_221813
Processing: run_20251116_221843
Processing: run_20251116_221914
Processing: run_20251116_221943
Processing: run_20251116_222014
Processing: run_20251116_222045
Processing: run_20251116_222115
  - Skipping p


--- Plot for group: PR=0, Seed=1337 ---



--- Plot for group: PR=0, Seed=42 ---



--- Plot for group: PR=1 ---



--- Summary Statistics (All Runs) ---


,run_name,avg_elapsed_time,std_dev,max_elapsed_time,ecmp_seed,use_precomputed_routes,folder_name
0,"cc:3, win:1, adapt:1, buf:1, size:1500, seed:2...",9.664376e+09,7.769491e+08,1.062045e+10,25,0,run_20251116_220401
1,"cc:3, win:1, adapt:1, buf:1, size:1500, seed:4...",NaN,NaN,NaN,42,0,run_20251116_220435
2,"cc:3, win:1, adapt:1, buf:1, size:1500, seed:1...",1.036239e+10,7.140043e+08,1.123105e+10,1337,0,run_20251116_220935
3,"cc:3, win:1, adapt:1, buf:1, size:1500, seed:2...",8.146363e+09,3.814332e+06,8.152382e+09,25,1,run_20251116_221008
4,"cc:3, win:1, adapt:1, buf:8, size:1500, seed:2...",9.664376e+09,7.769491e+08,1.062045e+10,25,0,run_20251116_221043
...,...,...,...,...,...,...,...
75,"cc:3, win:1, adapt:1, buf:8, size:1500, seed:1...",1.036239e+10,7.140043e+08,1.123105e+10,1337,0,run_20251117_152511
76,"cc:3, win:1, adapt:1, buf:8, size:1500, seed:2...",8.146363e+09,3.814332e+06,8.152382e+09,25,1,run_20251117_152601
77,"cc:10, win:1, adapt:1, buf:1, size:1500, seed:...",9.410255e+09,4.086282e+08,1.031622e+10,25,0,run_20251117_152650
78,"cc:10, win:1, adapt:1, buf:1, size:1500, seed:...",9.818571e+09,6.641117e+08,1.138899e+10,42,0,run_20251117_152736


In [4]:
pd.set_option('display.max_rows', 200)
results_df.sort_values(by='max_elapsed_time')

,run_name,avg_elapsed_time,std_dev,max_elapsed_time,ecmp_seed,use_precomputed_routes,folder_name
63,"cc:0, win:0, adapt:0, buf:8, size:1500, seed:2...",7.790726e+09,1.229634e+04,7.790736e+09,25,1,run_20251117_004646
59,"cc:0, win:0, adapt:0, buf:1, size:1500, seed:2...",7.790726e+09,1.229634e+04,7.790736e+09,25,1,run_20251117_003120
47,"cc:7, win:0, adapt:0, buf:8, size:1500, seed:2...",8.102117e+09,3.064880e+05,8.102687e+09,25,1,run_20251117_001144
43,"cc:7, win:0, adapt:0, buf:1, size:1500, seed:2...",8.102117e+09,3.064880e+05,8.102687e+09,25,1,run_20251116_235616
3,"cc:3, win:1, adapt:1, buf:1, size:1500, seed:2...",8.146363e+09,3.814332e+06,8.152382e+09,25,1,run_20251116_221008
67,"cc:3, win:1, adapt:1, buf:8, size:1500, seed:2...",8.146363e+09,3.814332e+06,8.152382e+09,25,1,run_20251117_101216
7,"cc:3, win:1, adapt:1, buf:8, size:1500, seed:2...",8.146363e+09,3.814332e+06,8.152382e+09,25,1,run_20251116_221225
15,"cc:10, win:1, adapt:1, buf:8, size:1500, seed:...",8.521709e+09,1.161980e+07,8.541354e+09,25,1,run_20251116_221642
11,"cc:10, win:1, adapt:1, buf:1, size:1500, seed:...",8.521709e+09,1.161980e+07,8.541354e+09,25,1,run_20251116_221433
55,"cc:7, win:1, adapt:1, buf:8, size:1500, seed:2...",8.577068e+09,1.553004e+07,8.596076e+09,25,1,run_20251117_001551
